# 高斯混合模型 GMM：概率版的 KMeans
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：03_无监督学习/聚类 | 源文件：高斯混合模型_GMM.py | 核心功能：软聚类、BIC/AIC 选模型、协方差类型对比、生成新样本

## 概述

如果 KMeans 是"硬聚类"（每个点属于且只属于一个簇），GMM 就是"软聚类"——每个点以不同概率属于各个簇。GMM 假设数据由 K 个高斯分布混合生成，用 EM（Expectation-Maximization）算法估计每个高斯的均值、协方差和混合权重。

与 KMeans 的关键区别：
- KMeans 只能发现球形簇，GMM 可以发现椭圆形簇（由协方差矩阵控制形状）
- KMeans 给出硬标签，GMM 给出概率分布
- GMM 可以生成新样本

脚本演示了四种协方差类型、BIC/AIC 选最优 K、软聚类概率输出和从模型生成新样本。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

## 数学原理

### 1. 高斯混合模型的概率框架

**代码对应**：`GaussianMixture(n_components=k)` 训练 GMM。

GMM 假设数据由 $K$ 个高斯分布混合生成：

$$P(\mathbf{x}) = \sum_{k=1}^{K}\pi_k \mathcal{N}(\mathbf{x}|\boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

其中 $\pi_k$ 为混合权重（$\sum_k \pi_k = 1$），$\boldsymbol{\mu}_k$ 和 $\boldsymbol{\Sigma}_k$ 为第 $k$ 个高斯分量的均值和协方差。

### 2. EM 算法求解

**E 步**（Expectation）：计算样本 $i$ 属于分量 $k$ 的**后验概率**（责任度）：

$$\gamma_{ik} = P(z_i = k|\mathbf{x}_i) = \frac{\pi_k \mathcal{N}(\mathbf{x}_i|\boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{\sum_{j=1}^{K}\pi_j \mathcal{N}(\mathbf{x}_i|\boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$$

**M 步**（Maximization）：用责任度更新参数：

$$\boldsymbol{\mu}_k^{\text{new}} = \frac{\sum_i \gamma_{ik}\mathbf{x}_i}{\sum_i \gamma_{ik}}, \quad \boldsymbol{\Sigma}_k^{\text{new}} = \frac{\sum_i \gamma_{ik}(\mathbf{x}_i - \boldsymbol{\mu}_k^{\text{new}})(\mathbf{x}_i - \boldsymbol{\mu}_k^{\text{new}})^T}{\sum_i \gamma_{ik}}$$

$$\pi_k^{\text{new}} = \frac{\sum_i \gamma_{ik}}{n}$$

EM 算法保证对数似然单调递增，但只能收敛到局部最优。

### 3. 对数似然

$$\ell(\boldsymbol{\theta}) = \sum_{i=1}^{n}\ln\left[\sum_{k=1}^{K}\pi_k \mathcal{N}(\mathbf{x}_i|\boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)\right]$$

### 4. 协方差类型

**代码对应**：`covariance_type` 参数（full、tied、diag、spherical）。

| 类型 | $\boldsymbol{\Sigma}_k$ 形状 | 参数数 | 适用场景 |
|------|---------------------------|--------|---------|
| full | 每个分量独立完整协方差 | $Kp(p+1)/2$ | 一般情况 |
| tied | 所有分量共享协方差 | $p(p+1)/2$ | 各簇形状相似 |
| diag | 对角协方差 | $Kp$ | 特征独立 |
| spherical | $\sigma_k^2\mathbf{I}$ | $K$ | 球形簇（类似 KMeans） |

### 5. 模型选择：BIC/AIC

**代码对应**：`gmm.bic(X)` 和 `gmm.aic(X)` 用于选择最优 $K$。

$$\text{BIC} = -2\ell(\hat{\boldsymbol{\theta}}) + d\ln n$$

$$\text{AIC} = -2\ell(\hat{\boldsymbol{\theta}}) + 2d$$

其中 $d$ 为参数总数。BIC 和 AIC 越小越好，BIC 对复杂模型惩罚更重。

### 6. GMM vs KMeans

KMeans 是 GMM 的特殊情况：当所有 $\boldsymbol{\Sigma}_k = \sigma^2\mathbf{I}$ 且 $\sigma \to 0$ 时，GMM 退化为 KMeans（硬分配）。

GMM 的优势：
- **软分配**：给出样本属于各簇的概率（不仅是标签）
- **椭圆形簇**：通过协方差矩阵捕捉非球形结构
- **概率模型**：可用 BIC/AIC 选择簇数

### 1. 生成数据

生成合成数据集，用于演示算法的核心行为。

In [ ]:
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=[1.0, 0.5, 1.5, 0.8], random_state=42)
X = StandardScaler().fit_transform(X)

### 2. 基础 GMM

运行 2. 基础 GMM 的代码，观察算法在该环节的行为。

In [ ]:
gmm = GaussianMixture(n_components=4, covariance_type="full", random_state=42, n_init=10)
gmm.fit(X)
labels = gmm.predict(X)
probs = gmm.predict_proba(X)

print("=== 高斯混合模型 (K=4) ===")
print(f"各簇样本数: {np.bincount(labels)}")
print(f"各簇权重 (mixing proportion): {gmm.weights_.round(4)}")
print(f"各簇均值:\n{gmm.means_.round(4)}")
print(f"BIC: {gmm.bic(X):.2f}")
# --- 输出结果 ---
print(f"AIC: {gmm.aic(X):.2f}")
print(f"对数似然: {gmm.score(X) * len(X):.2f}")
print(f"Silhouette: {silhouette_score(X, labels):.4f}")
print(f"ARI: {adjusted_rand_score(y_true, labels):.4f}")

### 3. 协方差类型

运行 3. 协方差类型 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 协方差类型对比 ===")
for cov_type in ["full", "tied", "diag", "spherical"]:
    gmm_c = GaussianMixture(n_components=4, covariance_type=cov_type, random_state=42, n_init=10)
    gmm_c.fit(X)
    labels_c = gmm_c.predict(X)
# --- 继续 ---
    sil = silhouette_score(X, labels_c)
    n_params = gmm_c._n_parameters()
    print(f"  {cov_type:>10}: BIC={gmm_c.bic(X):>10.2f}, AIC={gmm_c.aic(X):>10.2f}, "
          f"参数数={n_params}, Silhouette={sil:.4f}")
print("\n  full: 完整协方差（参数最多，最灵活）")
# --- 输出结果 ---
print("  tied: 所有簇共享同一协方差矩阵")
print("  diag: 对角协方差（各特征独立）")
print("  spherical: 各簇方差相同（最简单）")

### 4. 用 BIC/AIC 选簇数

运行 4. 用 BIC/AIC 选簇数 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== BIC/AIC 选最优 K ===")
bic_scores = []
aic_scores = []
for k in range(1, 10):
    gmm_k = GaussianMixture(n_components=k, covariance_type="full", random_state=42, n_init=10)
# --- 训练模型 ---
    gmm_k.fit(X)
    bic_scores.append(gmm_k.bic(X))
    aic_scores.append(gmm_k.aic(X))
    print(f"  K={k}: BIC={gmm_k.bic(X):>10.2f}, AIC={gmm_k.aic(X):>10.2f}")

best_k_bic = np.argmin(bic_scores) + 1
best_k_aic = np.argmin(aic_scores) + 1
print(f"\n  BIC 最优 K: {best_k_bic}")
print(f"  AIC 最优 K: {best_k_aic}")

### 5. 软聚类（概率分配）

运行 5. 软聚类（概率分配） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 软聚类（概率分配）===")
gmm_final = GaussianMixture(n_components=4, random_state=42, n_init=10)
gmm_final.fit(X)
probs = gmm_final.predict_proba(X)
print("前 5 个样本的簇概率:")
# --- 循环处理 ---
for i in range(5):
    prob_str = ", ".join(f"p{k}={p:.3f}" for k, p in enumerate(probs[i]))
    print(f"  样本{i+1}: {prob_str} → 簇 {labels[i]}")

### 6. 生成新样本

运行 6. 生成新样本 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 从 GMM 生成新样本 ===")
X_new, y_new = gmm_final.sample(5)
print("生成的 5 个新样本:")
for i in range(5):
    print(f"  样本{i+1}: {X_new[i].round(4)} (来自簇 {y_new[i]})")

print("\n=== GMM 要点 ===")
print("- 与 KMeans 的区别：GMM 是软聚类（概率分配），KMeans 是硬聚类（确定分配）")
print("- 簇可以是椭圆形（由协方差矩阵控制形状）")
print("- 用 EM 算法（期望最大化）迭代优化")
print("- BIC 惩罚模型复杂度，通常选 BIC 最小的 K")
# --- 输出结果 ---
print("- AIC 倾向选择更复杂的模型（更大的 K）")
print("- 对初始值敏感，建议增加 n_init")

## 关键代码解释

### BIC/AIC 选最优 K

```python
for k in range(1, 10):
    gmm_k = GaussianMixture(n_components=k, random_state=42, n_init=10)
    bic_scores.append(gmm_k.bic(X))
```

BIC（贝叶斯信息准则）和 AIC（赤池信息准则）都在对数似然的基础上惩罚模型复杂度。BIC 惩罚更重，倾向选择更简单的模型。**BIC 最小的 K 通常是好选择**。

### 四种协方差类型

```python
for cov_type in ["full", "tied", "diag", "spherical"]:
```

- **full**：每个簇有自己的完整协方差矩阵（最灵活，参数最多）
- **tied**：所有簇共享同一个协方差矩阵
- **diag**：对角协方差（特征独立）
- **spherical**：各方向等方差（最简单，接近 KMeans）

### 软聚类

```python
probs = gmm_final.predict_proba(X)
```

每个样本得到一个概率向量，表示它属于各个簇的概率。如果某个样本的 [0.45, 0.40, 0.10, 0.05] 意味着它在簇 0 和簇 1 之间"犹豫不决"——这比 KMeans 的硬标签提供了更多信息。

## 使用示例

```python
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=4, covariance_type="full", n_init=10)
labels = gmm.fit_predict(X)
probs = gmm.predict_proba(X)
```

## 注意事项

1. **对初始值敏感**：
_init=10 运行多次取最优
2. **BIC 比 AIC 更保守**：倾向于选更少的簇
3. **full 协方差在高维时参数爆炸**：d 维有 O(d²) 个参数
4. **EM 可能收敛到局部最优**
5. **可以生成新样本**：gmm.sample(n)

## 总结与延伸

以上代码展示了 **高斯混合模型 GMM** 的完整流程。

**解读要点**：
- 关注 **轮廓系数（Silhouette Score）**，越接近 1 越好
- 观察聚类中心是否与数据分布吻合
- 对比不同 K 值的聚类效果

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **变分高斯混合模型（VBGMM）**：贝叶斯版本，自动确定有效成分数
- **GMM 用于异常检测**：低似然度的点可能是异常值
- **GMM 用于语音识别**：说话人识别中的经典方法
- **贝叶斯高斯混合**：BayesianGaussianMixture，自动确定有效簇数
